In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    def __init__(self, d_model, head_nums):
        super().__init__()
        self.d_model = d_model
        self.head_nums = head_nums
        self.head_dim = d_model // head_nums

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.out = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, D = x.shape

        q = self.q(x)
        k = self.q(k)
        v = self.q(v)
        
        # (B, T, D) --- > (B. T, H, Hd) ---- > (B, H, T, Hd)
        # (a, b) @ (b,c) = (a, c)
        # (B, T, D) @ (B, T, D)
        #(T, D) @ (D, T)
        q = q.view(B, T, self.head_nums, self.head_dim).trasnpose(1, 2)
        k = k.view(B, T, self.head_nums, self.head_dim).trasnpose(1, 2)
        v = v.view(B, T, self.head_nums, self.head_dim).trasnpose(1, 2)
        # (B, H, T, Hd) @ (B, H, Hd,T ) = (B, H, T, T)
        attn_score = q @ k.transpose(-2, -1)
        score = attn_score / self.head_dim ** 0.5
        mask = torch.tril(torch.ones(T, T)).bool()
        score = score.masked(~mask, float("-inf"))
        weight = F.softmax(score, dim=-1)
        # (B, H, T, T) @ (B, H, T, Hd) = (B, H, T, Hd)
        out = weight @ v
        out = out.transpose(1, 2).contiguous() # (B, H, T, Hd) --- > (B, T, H, Hd)
        out = out.view(B, T, D) # (B, T, H, Hd) ---> (B, T, D) out.reshape()
        return self.out(out)

# Decision Tree

Decision Tree — bu **supervised learning** algoritmi bo‘lib, u data ni ketma-ket savollar orqali bo‘ladi.

---

## 1. Intuition

- har bir tugun bir savol beradi
- har bir branch bir qaror yo‘lini bildiradi
- `leaf` — yakuniy javob

> Maqsad: ma’lumotni tobora tozaroq guruhlarga ajratish.

---

## 2. Main terms

| Term | Meaning |
|------|---------|
| feature | input column |
| label | target column |
| node | decision point |
| root | boshlang‘ich tugun |
| leaf | final node |
| depth | tree chuqurligi |
| impurity | aralashlik darajasi |
| pruning | keraksiz branchlarni kesish |

---

## 3. Math

$$
Entropy(S) = - \sum_{i=1}^{c} p_i \log_2(p_i)
$$

$$
Gini(S) = 1 - \sum_{i=1}^{c} p_i^2
$$

$$
Information\ Gain = Entropy(parent) - \sum_k \frac{|S_k|}{|S|} Entropy(S_k)
$$